# Pipeline: Landing → Bronze

Este notebook implementa el pipeline de ingestión desde la **landing zone** hacia la capa **Bronze**.

### ¿Qué hace este pipeline?
1. **Extrae** los archivos CSV desde el volume `/Volumes/landing/origin_system/raw_data/`
2. **Valida** las claves naturales (unicidad + no nulidad) de cada tabla
3. **Agrega columnas de sistema** (`PK_HASH`, `R_HASH`, `CREATED_AT`, `UPDATED_AT`, `DELETED`, `DELETED_AT`)
4. **Carga** cada tabla en Bronze mediante **UPSERT** (Merge en Delta Lake)

### Tablas generadas
| Tabla Bronze                        | Archivo Fuente      | PK              |
|-------------------------------------|---------------------|-----------------|
| `bronze.origin_system.venta`        | `Venta.csv`         | `IdVenta`       |
| `bronze.origin_system.clientes`     | `Clientes.csv`      | `ID`            |
| `bronze.origin_system.productos`    | `Productos.csv`     | `ID_PRODUCTO`   |
| `bronze.origin_system.sucursales`   | `Sucursales.csv`    | `ID`            |
| `bronze.origin_system.empleados`    | `Empleados.csv`     | `ID_EMPLEADO`   |
| `bronze.origin_system.compra`       | `Compra.csv`        | `IdCompra`      |
| `bronze.origin_system.gasto`        | `Gasto.csv`         | `IdGasto`       |
| `bronze.origin_system.tipos_gasto`  | `TiposDeGasto.csv`  | `IdTipoGasto`   |
| `bronze.origin_system.canal_venta`  | `CanalDeVenta.csv`  | `IdCanal`       |
| `bronze.origin_system.proveedores`  | `Proveedores.csv`   | `IdProveedor`   |
| `bronze.origin_system.calendario`   | `Calendario.csv`    | `Fecha`         |

### Patrón UPSERT aplicado
- **Primera ejecución**: escribe la tabla Delta completa con `saveAsTable`
- **Ejecuciones posteriores**: realiza un `MERGE` por `PK_HASH` → actualiza registros existentes, inserta los nuevos

> ⚠️ **Prerequisito**: Ejecutar los scripts en `configs/` antes de correr este pipeline.

### Imports de librerías

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, lit, md5, concat_ws
from pyspark.sql.types import BooleanType, TimestampType
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import DeltaTable
from typing import Dict, List
import logging

### Configuración de la sesión Spark

In [0]:
spark = (
    SparkSession.builder
    .appName('LandingToBronzeApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

### Parámetros del pipeline

In [0]:
# Catálogo fuente (landing zone — volumes)
LANDING_CATALOG = "landing"
LANDING_SCHEMA  = "origin_system"
LANDING_VOLUME  = "raw_data"
LANDING_PATH    = f"/Volumes/{LANDING_CATALOG}/{LANDING_SCHEMA}/{LANDING_VOLUME}"

# Catálogo destino (bronze — Delta tables)
BRONZE_CATALOG = "bronze"
BRONZE_SCHEMA  = "origin_system"

# Zona horaria para columnas de sistema
TIMEZONE = "America/Argentina/Buenos_Aires"

print(f"Landing path : {LANDING_PATH}")
print(f"Bronze target: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")

---
## Funciones del pipeline

> Estas funciones encapsulan la lógica reutilizable del ETL.

#### Función 1: `validate_natural_key`
Verifica que la clave primaria de cada DataFrame sea **única y no nula**.

In [0]:
def validate_natural_key(dataframe_and_key: Dict[str, tuple]) -> None:
    """
    Valida que la clave natural de cada DataFrame sea única y no nula.
    
    Args:
        dataframe_and_key: diccionario { 'nombre_tabla': (df, [col_pk1, col_pk2...]) }
    
    Raises:
        ValueError: si alguna clave tiene nulos o duplicados.
    """
    validated = []

    for table_name, (df, key_columns) in dataframe_and_key.items():
        null_condition  = ' OR '.join([f"{c} IS NULL" for c in key_columns])
        null_count      = df.filter(null_condition).count()
        duplicate_count = df.groupBy(*key_columns).count().filter("count > 1").count()

        if null_count > 0 or duplicate_count > 0:
            raise ValueError(
                f"[{table_name}] Clave inválida {key_columns}: "
                f"{null_count} nulos, {duplicate_count} duplicados."
            )
        else:
            logger.info(f"[{table_name}] ✅ Clave {key_columns} válida.")
            validated.append(table_name)

    logger.info(f"Validación completada para: {validated}")

#### Función 2: `prepare_bronze_dataframe`
Agrega las **columnas de sistema** necesarias para la capa Bronze:
- Normaliza nombres de columnas a **MAYÚSCULAS**
- `PK_HASH`: hash MD5 de la clave primaria (usado en el MERGE)
- `R_HASH`: hash MD5 del resto de las columnas (detecta cambios)
- `CREATED_AT` / `UPDATED_AT`: timestamps de auditoría
- `DELETED` / `DELETED_AT`: soporte para soft delete

In [0]:
def prepare_bronze_dataframe(df: DataFrame, pk_columns: List[str]) -> DataFrame:
    """
    Prepara un DataFrame para la capa Bronze agregando columnas de sistema.

    Args:
        df          : DataFrame fuente leído desde la landing zone.
        pk_columns  : Lista de columnas que forman la clave primaria natural.

    Returns:
        DataFrame con columnas de sistema añadidas.
    """
    # 1. Normalizar todos los nombres de columna a MAYÚSCULAS
    processed_df = df
    for column in df.columns:
        processed_df = processed_df.withColumnRenamed(column, column.upper())

    # 2. Versión uppercase de las columnas PK
    pk_upper  = [c.upper() for c in pk_columns]
    all_upper = [c.upper() for c in df.columns]
    r_columns = [c for c in all_upper if c not in pk_upper]

    # 3. Agregar columnas de sistema
    return (
        processed_df
        .withColumn("PK_HASH",    md5(concat_ws("|", *[col(c) for c in pk_upper])))
        .withColumn("R_HASH",     md5(concat_ws("|", *[col(c) for c in r_columns])))
        .withColumn("CREATED_AT", from_utc_timestamp(current_timestamp(), TIMEZONE))
        .withColumn("UPDATED_AT", from_utc_timestamp(current_timestamp(), TIMEZONE))
        .withColumn("DELETED",    lit(False).cast(BooleanType()))
        .withColumn("DELETED_AT", lit(None).cast(TimestampType()))
    )

#### Función 3: `upsert_to_bronze`
Implementa el **UPSERT** (INSERT + UPDATE) usando Delta Lake MERGE.

- **Si la tabla no existe** → `saveAsTable` (creación inicial)
- **Si la tabla existe** → `MERGE` por `PK_HASH`
  - `WHEN MATCHED` → actualiza todas las columnas **excepto** `CREATED_AT` y `R_HASH` (la timestamp de creación nunca cambia)
  - `WHEN NOT MATCHED` → inserta la fila nueva completa

In [0]:
def upsert_to_bronze(df: DataFrame, table_name: str) -> None:
    """
    Realiza un UPSERT de un DataFrame hacia una tabla Delta en Bronze.

    Estrategia:
      - Si la tabla NO existe → crea la tabla con saveAsTable.
      - Si la tabla YA existe → ejecuta MERGE por PK_HASH:
          · WHEN MATCHED     → actualiza columnas de datos y UPDATED_AT
          · WHEN NOT MATCHED → inserta la fila nueva

    Args:
        df         : DataFrame con columnas de sistema ya aplicadas (prepare_bronze_dataframe).
        table_name : Nombre completo de la tabla destino (ej: 'bronze.origin_system.venta').
    """
    # Columnas que NO se actualizan en un MATCHED (las de control de creación)
    exclude_on_update = {'CREATED_AT', 'R_HASH'}

    try:
        # Intento de creación inicial
        df.write.format("delta").saveAsTable(table_name)
        logger.info(f"[{table_name}] ✅ Tabla creada por primera vez.")

    except AnalysisException as err:
        if "TABLE_OR_VIEW_ALREADY_EXISTS" in err.getErrorClass():
            logger.info(f"[{table_name}] Tabla existente — ejecutando MERGE (upsert)...")

            target_table = DeltaTable.forName(spark, table_name)

            # Columnas a actualizar en WHEN MATCHED
            update_cols = {
                f"target.{c}": f"source.{c}"
                for c in df.columns
                if c not in exclude_on_update
            }

            (
                target_table.alias("target")
                .merge(
                    df.alias("source"),
                    "target.PK_HASH = source.PK_HASH"
                )
                .whenMatchedUpdate(set=update_cols)
                .whenNotMatchedInsertAll()
                .execute()
            )

            logger.info(f"[{table_name}] ✅ MERGE completado.")
        else:
            raise err

---
## Paso 1 — Extracción desde Landing Zone

Leemos los archivos CSV desde el volume de landing. Cada archivo tiene su configuración
de separador específica según el sistema de origen.

In [0]:
# Opciones de lectura comunes para archivos con separador coma
csv_opts_comma = {"header": "true", "inferSchema": "true", "sep": ",", "encoding": "UTF-8"}

# Opciones de lectura para archivos con separador punto y coma (sistema legado)
csv_opts_semicolon = {"header": "true", "inferSchema": "true", "sep": ";", "encoding": "ISO-8859-1"}

def read_csv(filename: str, opts: dict) -> DataFrame:
    """Lee un CSV desde la landing zone con las opciones dadas."""
    path = f"{LANDING_PATH}/{filename}"
    return spark.read.format("csv").options(**opts).load(path)

# --- Lectura de todos los archivos fuente ---
df_venta        = read_csv("Venta.csv",        csv_opts_comma)
df_clientes     = read_csv("Clientes.csv",     csv_opts_semicolon)
df_productos    = read_csv("Productos.csv",    csv_opts_comma)
df_sucursales   = read_csv("Sucursales.csv",   csv_opts_semicolon)
df_empleados    = read_csv("Empleados.csv",    csv_opts_comma)
df_compra       = read_csv("Compra.csv",       csv_opts_comma)
df_gasto        = read_csv("Gasto.csv",        csv_opts_comma)
df_tipos_gasto  = read_csv("TiposDeGasto.csv", csv_opts_comma)
df_canal_venta  = read_csv("CanalDeVenta.csv", csv_opts_comma)
df_proveedores  = read_csv("Proveedores.csv",  csv_opts_comma)
df_calendario   = read_csv("Calendario.csv",   csv_opts_comma)

print("✅ Todos los archivos CSV leídos desde la landing zone.")

---
## Paso 2 — Verificación de schemas

Inspeccionamos los esquemas de los DataFrames para asegurarnos de que la inferencia
de tipos fue correcta antes de avanzar.

In [0]:
print("=" * 50)
print("SCHEMA: Venta")
df_venta.printSchema()

print("=" * 50)
print("SCHEMA: Clientes")
df_clientes.printSchema()

print("=" * 50)
print("SCHEMA: Productos")
df_productos.printSchema()

print("=" * 50)
print("SCHEMA: Sucursales")
df_sucursales.printSchema()

print("=" * 50)
print("SCHEMA: Empleados")
df_empleados.printSchema()

print("=" * 50)
print("SCHEMA: Compra")
df_compra.printSchema()

print("=" * 50)
print("SCHEMA: Gasto")
df_gasto.printSchema()

print("=" * 50)
print("SCHEMA: TiposDeGasto")
df_tipos_gasto.printSchema()

print("=" * 50)
print("SCHEMA: CanalDeVenta")
df_canal_venta.printSchema()

print("=" * 50)
print("SCHEMA: Proveedores")
df_proveedores.printSchema()

print("=" * 50)
print("SCHEMA: Calendario")
df_calendario.printSchema()

---
## Paso 3 — Validación de claves naturales

Antes de cargar en Bronze, verificamos que cada tabla tenga una **clave primaria válida**:
- Sin valores nulos
- Sin duplicados

> Si esta celda lanza un `ValueError`, hay un problema de calidad de datos en el archivo fuente.

In [0]:
validate_natural_key({
    "venta"       : (df_venta,       ["IdVenta"]),
    "clientes"    : (df_clientes,    ["ID"]),
    "productos"   : (df_productos,   ["ID_PRODUCTO"]),
    "sucursales"  : (df_sucursales,  ["ID"]),
    "empleados"   : (df_empleados,   ["ID_empleado"]),
    "compra"      : (df_compra,      ["IdCompra"]),
    "gasto"       : (df_gasto,       ["IdGasto"]),
    "tipos_gasto" : (df_tipos_gasto, ["IdTipoGasto"]),
    "canal_venta" : (df_canal_venta, ["IdCanal"]),
    "proveedores" : (df_proveedores, ["IdProveedor"]),
    "calendario"  : (df_calendario,  ["Fecha"]),
})

print("\n✅ Todas las claves naturales son válidas. Avanzando a Bronze.")

---
## Paso 4 — Preparación de DataFrames con columnas de sistema

Aplicamos `prepare_bronze_dataframe` a cada tabla para agregar `PK_HASH`, `R_HASH`,
timestamps de auditoría y flags de soft delete.

In [0]:
bronze_venta       = prepare_bronze_dataframe(df_venta,       ["IdVenta"])
bronze_clientes    = prepare_bronze_dataframe(df_clientes,    ["ID"])
bronze_productos   = prepare_bronze_dataframe(df_productos,   ["ID_PRODUCTO"])
bronze_sucursales  = prepare_bronze_dataframe(df_sucursales,  ["ID"])
bronze_empleados   = prepare_bronze_dataframe(df_empleados,   ["ID_empleado"])
bronze_compra      = prepare_bronze_dataframe(df_compra,      ["IdCompra"])
bronze_gasto       = prepare_bronze_dataframe(df_gasto,       ["IdGasto"])
bronze_tipos_gasto = prepare_bronze_dataframe(df_tipos_gasto, ["IdTipoGasto"])
bronze_canal_venta = prepare_bronze_dataframe(df_canal_venta, ["IdCanal"])
bronze_proveedores = prepare_bronze_dataframe(df_proveedores, ["IdProveedor"])
bronze_calendario  = prepare_bronze_dataframe(df_calendario,  ["Fecha"])

print("✅ DataFrames preparados con columnas de sistema.")
print("\nEjemplo — schema de bronze_venta:")
bronze_venta.printSchema()

---
## Paso 5 — Carga a Bronze (UPSERT)

Ejecutamos el upsert para cada tabla. El orden de carga respeta primero las tablas
de maestros (dimensiones) y luego las tablas de hechos (transacciones).

#### 5a. Tablas de maestros / dimensiones

In [0]:
upsert_to_bronze(bronze_clientes,    f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.clientes")
upsert_to_bronze(bronze_productos,   f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.productos")
upsert_to_bronze(bronze_sucursales,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.sucursales")
upsert_to_bronze(bronze_empleados,   f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.empleados")
upsert_to_bronze(bronze_canal_venta, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.canal_venta")
upsert_to_bronze(bronze_proveedores, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.proveedores")
upsert_to_bronze(bronze_tipos_gasto, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.tipos_gasto")
upsert_to_bronze(bronze_calendario,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.calendario")

print("\n✅ Tablas de maestros cargadas en Bronze.")

#### 5b. Tablas de hechos / transacciones

In [0]:
upsert_to_bronze(bronze_venta,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.venta")
upsert_to_bronze(bronze_compra, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.compra")
upsert_to_bronze(bronze_gasto,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.gasto")

print("\n✅ Tablas de hechos cargadas en Bronze.")

---
## Paso 6 — Verificación de carga

Consultamos el catálogo de Bronze para confirmar que todas las tablas fueron creadas/actualizadas correctamente.

In [0]:
print(f"Tablas en {BRONZE_CATALOG}.{BRONZE_SCHEMA}:")
spark.sql(f"SHOW TABLES IN {BRONZE_CATALOG}.{BRONZE_SCHEMA}").show()

# Conteo rápido de registros por tabla
tablas = [
    "venta", "clientes", "productos", "sucursales", "empleados",
    "compra", "gasto", "tipos_gasto", "canal_venta", "proveedores", "calendario"
]

print("\nConteo de registros por tabla:")
print("-" * 45)
for tabla in tablas:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{tabla}").count()
    print(f"  {tabla:<20} : {count:>10,} registros")
print("-" * 45)

---
## Pipeline completado ✅

Los datos de la landing zone fueron procesados y cargados en Bronze.

**Próximo paso**: ejecutar el pipeline `sales_pipeline_silver` que toma los datos de
`bronze.origin_system` y genera la tabla enriquecida `silver.sales.total_sales`.